# Step 2: Feature Engineering — KBS Submission (v2 Bug-Fixed)

**v2 fixes three bugs that caused 0-row exports:**
1. All-NaN columns (buoy SSH) now audited and dropped before engineering.
2. Rolling window=1 removed (std of 1 sample is always NaN, ddof=1).
3. dropna() is now targeted on essential columns + ffill/bfill boundary NaNs.

### 0 — Mount Google Drive (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted.")

Mounted at /content/drive
✓ Google Drive mounted.


### 1 — Imports & Constants

In [2]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import timedelta

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_PATH  = ('/content/drive/MyDrive/KBS_Paper/Outputs/'
               '1_Predictor_Selection_KBS/'
               'master_dataset_causally_optimized_v5.csv')
OUTPUT_DIR  = ('/content/drive/MyDrive/KBS_Paper/Outputs/'
               '2_Feature_Engineering_KBS/')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR}")

# ── Physical & modelling constants ────────────────────────────────────────────
G                    = 9.81          # m/s² gravitational acceleration
TIME_STEP_HOURS      = 3             # dataset resolution (3-h intervals)
FORECAST_LEAD_STEPS  = 1             # 1 step ahead  =  +3 h
TEMPORAL_GAP_HOURS   = 24            # anti-leakage purge window (hours)
LAG_STEPS            = list(range(1, 11))          # lags 1–10 steps (3h–30h)
# NOTE: window=1 is intentionally excluded.
#   (a) mean/max of window=1 equals the raw value itself → redundant with lag_0.
#   (b) std of window=1 is always NaN (pandas ddof=1) → would wipe every row
#       via downstream dropna(). Window starts at 2 steps = 6 h.
ROLLING_STEP_SIZES   = [2, 4, 8]                  # steps: 6h, 12h, 24h

# Source point_ids present in the master dataset
OFFSHORE_IDS = ['offshore_34', 'offshore_56', 'offshore_26',
                'offshore_79', 'offshore_46']
BUOY_MAIN_ID = 'buoy_main'
BUOY_OOS_ID  = 'buoy_oos'

# Raw variable names (long-format column names from Step 1 output)
RAW_VARS = ['hm0', 'tp', 'mdir', 'windspeed', 'winddirection', 'ssh']

# Generic names for target-buoy columns (same mapping applied to both buoys)
BUOY_RENAME = {
    'hm0'          : 'target_buoy_hs',
    'tp'           : 'target_buoy_tp',
    'mdir'         : 'target_buoy_mdir',
    'windspeed'    : 'target_buoy_windspeed',
    'winddirection': 'target_buoy_winddir',
    'ssh'          : 'target_buoy_ssh',
}

# Publication figure style
def set_pub_style():
    sns.set_style("ticks")
    plt.rcParams.update({
        'font.family'       : 'serif',
        'font.serif'        : ['Times New Roman', 'Arial'],
        'font.size'         : 11,
        'axes.linewidth'    : 0.8,
        'xtick.major.width' : 0.8,
        'ytick.major.width' : 0.8,
    })

def save_fig(fig, stem: str, dpi: int = 600):
    """Save PNG + TIFF at journal quality and close the figure."""
    for ext in ('png', 'tiff'):
        p = os.path.join(OUTPUT_DIR, f"{stem}.{ext}")
        fig.savefig(p, dpi=dpi, bbox_inches='tight', facecolor='white')
        print(f"    ✓ Saved → {p}")
    plt.close(fig)

print("✓ Constants & helpers defined.")

✓ Output directory ready: /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/
✓ Constants & helpers defined.


### 2 — Load Master Dataset

In [3]:
print("\n" + "="*60)
print("STEP A: LOADING MASTER DATASET")
print("="*60)

df_raw = pd.read_csv(INPUT_PATH)
df_raw['time'] = pd.to_datetime(df_raw['time'], format='mixed', utc=False)
df_raw['time'] = df_raw['time'].dt.tz_localize(None)   # strip tz if present
df_raw.set_index('time', inplace=True)
df_raw.sort_index(inplace=True)

print(f"  Raw master dataset loaded.  Shape: {df_raw.shape}")
print(f"  point_id distribution:\n{df_raw['point_id'].value_counts()}")
print(f"  Columns: {list(df_raw.columns)}")


STEP A: LOADING MASTER DATASET
  Raw master dataset loaded.  Shape: (125972, 10)
  point_id distribution:
point_id
offshore_46    20453
offshore_56    20453
offshore_34    20453
offshore_26    20453
offshore_79    20453
buoy_main      18098
buoy_oos        5609
Name: count, dtype: int64
  Columns: ['point_id', 'lat', 'lon', 'hm0', 'tp', 'mdir', 'windspeed', 'winddirection', 'depth', 'ssh']


### 3 — Extract Offshore History (wide pivot, independent of buoy timelines)

In [4]:
print("\n" + "="*60)
print("STEP B: EXTRACTING OFFSHORE HISTORY (WIDE FORMAT)")
print("="*60)

df_offshore_long = df_raw[df_raw['point_id'].isin(OFFSHORE_IDS)].copy()

# Keep only numeric variables that actually exist in the dataframe
vars_available = [v for v in RAW_VARS if v in df_offshore_long.columns]

# Pivot: index=time, columns=point_id × variable → offshore_34_hs, etc.
df_offshore_wide = df_offshore_long.pivot_table(
    index='time',
    columns='point_id',
    values=vars_available,
    aggfunc='mean'          # harmless: each (time, point_id) is unique
)

# Flatten MultiIndex columns: (variable, point_id) → point_id_variable
df_offshore_wide.columns = [
    f"{pid}_{var.replace('hm0', 'hs')}"           # hm0 → hs everywhere
    for var, pid in df_offshore_wide.columns
]
df_offshore_wide.sort_index(inplace=True)

print(f"  Offshore wide shape: {df_offshore_wide.shape}")
print(f"  Offshore columns (first 12): {list(df_offshore_wide.columns[:12])}")
print(f"  Offshore time range: {df_offshore_wide.index.min()} → {df_offshore_wide.index.max()}")
del df_offshore_long
gc.collect()


STEP B: EXTRACTING OFFSHORE HISTORY (WIDE FORMAT)
  Offshore wide shape: (20453, 30)
  Offshore columns (first 12): ['offshore_26_hs', 'offshore_34_hs', 'offshore_46_hs', 'offshore_56_hs', 'offshore_79_hs', 'offshore_26_mdir', 'offshore_34_mdir', 'offshore_46_mdir', 'offshore_56_mdir', 'offshore_79_mdir', 'offshore_26_ssh', 'offshore_34_ssh']
  Offshore time range: 2018-01-01 00:00:00 → 2024-12-31 12:00:00


0

### 4 — Build Train/Val Base  (buoy_main ⊕ offshore history)

In [5]:
print("\n" + "="*60)
print("STEP C: BUILDING TRAIN/VAL BASE (buoy_main)")
print("="*60)

df_buoy_main = (
    df_raw[df_raw['point_id'] == BUOY_MAIN_ID]
    .drop(columns=['point_id', 'lat', 'lon', 'depth'], errors='ignore')
    .copy()
)

# Rename raw buoy vars to generic names; keep only vars that exist
rename_map_train = {k: v for k, v in BUOY_RENAME.items()
                    if k in df_buoy_main.columns}
df_buoy_main.rename(columns=rename_map_train, inplace=True)

# Left-join with offshore wide on shared time index
df_train_base = df_buoy_main.join(df_offshore_wide, how='left')
df_train_base.sort_index(inplace=True)

print(f"  buoy_main rows (before offshore join): {len(df_buoy_main)}")
print(f"  df_train_base shape: {df_train_base.shape}")
print(f"  Train time range: {df_train_base.index.min()} → {df_train_base.index.max()}")


STEP C: BUILDING TRAIN/VAL BASE (buoy_main)
  buoy_main rows (before offshore join): 18098
  df_train_base shape: (18098, 36)
  Train time range: 2018-07-10 06:00:00 → 2024-09-18 09:00:00


### 5 — Build OOS Base  (buoy_oos ⊕ offshore history)

In [6]:
print("\n" + "="*60)
print("STEP D: BUILDING OOS BASE (buoy_oos)")
print("="*60)

df_buoy_oos = (
    df_raw[df_raw['point_id'] == BUOY_OOS_ID]
    .drop(columns=['point_id', 'lat', 'lon', 'depth'], errors='ignore')
    .copy()
)

# Use THE SAME generic rename map so both bases share identical column names
rename_map_oos = {k: v for k, v in BUOY_RENAME.items()
                  if k in df_buoy_oos.columns}
df_buoy_oos.rename(columns=rename_map_oos, inplace=True)

df_oos_base = df_buoy_oos.join(df_offshore_wide, how='left')
df_oos_base.sort_index(inplace=True)

print(f"  buoy_oos rows (before offshore join): {len(df_buoy_oos)}")
print(f"  df_oos_base shape: {df_oos_base.shape}")
print(f"  OOS time range: {df_oos_base.index.min()} → {df_oos_base.index.max()}")

del df_buoy_main, df_buoy_oos, df_raw
gc.collect()


STEP D: BUILDING OOS BASE (buoy_oos)
  buoy_oos rows (before offshore join): 5609
  df_oos_base shape: (5609, 36)
  OOS time range: 2022-10-18 09:00:00 → 2024-09-18 09:00:00


0

### 6 — 24-Hour Anti-Leakage Protocol

In [7]:
print("\n" + "="*60)
print("STEP E: 24-HOUR ANTI-LEAKAGE CUTOFF")
print("="*60)

# Find the first valid (non-NaT) timestamp in OOS
first_oos_ts = df_oos_base.index.dropna().min()
print(f"  First valid OOS timestamp : {first_oos_ts}")

# Subtract exactly 24 hours
# Physical rationale:
#   • 12.4-h semi-diurnal (M2) tidal autocorrelation → a 24-h gap ensures
#     ≥1 full tidal cycle separates train from OOS, purging tidal memory.
#   • 24-h thermal land–sea breeze inertia → the diurnal wind forcing cycle
#     cannot "bleed" signal from training rows into OOS predictions.
leakage_cutoff = first_oos_ts - timedelta(hours=TEMPORAL_GAP_HOURS)
print(f"  Anti-leakage cutoff (T_oos - 24h): {leakage_cutoff}")

# Drop any train rows AFTER the cutoff (strict inequality)
n_before = len(df_train_base)
df_train_base = df_train_base[df_train_base.index <= leakage_cutoff]
n_after = len(df_train_base)
print(f"  Train rows: {n_before} → {n_after}  "
      f"({n_before - n_after} rows purged beyond cutoff)")
print(f"  Train time range after cutoff: "
      f"{df_train_base.index.min()} → {df_train_base.index.max()}")


STEP E: 24-HOUR ANTI-LEAKAGE CUTOFF
  First valid OOS timestamp : 2022-10-18 09:00:00
  Anti-leakage cutoff (T_oos - 24h): 2022-10-17 09:00:00
  Train rows: 18098 → 12482  (5616 rows purged beyond cutoff)
  Train time range after cutoff: 2018-07-10 06:00:00 → 2022-10-17 09:00:00


### 7 — Feature Engineering Functions (Modular)

In [8]:
print("\n" + "="*60)
print("STEP F: FEATURE ENGINEERING FUNCTIONS")
print("="*60)

def add_lag_features(df: pd.DataFrame,
                     cols: list,
                     lag_steps: list) -> pd.DataFrame:
    """
    Append lag features for each column in `cols`.

    Parameters
    ----------
    df        : DataFrame with DatetimeIndex (3-h resolution)
    cols      : list of column names to lag
    lag_steps : list of integer step offsets (e.g. [1..10])

    Returns
    -------
    DataFrame with lag columns appended (original columns unchanged).
    """
    lag_frames = []
    for col in cols:
        if col not in df.columns:
            continue
        for step in lag_steps:
            hours = step * TIME_STEP_HOURS
            lag_frames.append(
                df[col].shift(step).rename(f"{col}_lag_{hours}h")
            )
    if lag_frames:
        df = pd.concat([df] + lag_frames, axis=1)
    return df


def add_rolling_features(df: pd.DataFrame,
                         cols: list,
                         window_steps: list) -> pd.DataFrame:
    """
    Append rolling mean, std, and max for each column.

    Parameters
    ----------
    df           : DataFrame with DatetimeIndex (3-h resolution)
    cols         : list of column names
    window_steps : window sizes in data steps (e.g. [1,2,4,8] → 3h,6h,12h,24h)
    """
    roll_frames = []
    for col in cols:
        if col not in df.columns:
            continue
        for steps in window_steps:
            hours = steps * TIME_STEP_HOURS
            rw = df[col].rolling(window=steps, min_periods=1)
            roll_frames.append(rw.mean().rename(f"{col}_roll_mean_{hours}h"))
            roll_frames.append(rw.std().rename(f"{col}_roll_std_{hours}h"))
            roll_frames.append(rw.max().rename(f"{col}_roll_max_{hours}h"))
    if roll_frames:
        df = pd.concat([df] + roll_frames, axis=1)
    return df


def add_cyclical_time_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Encode hour-of-day and day-of-year as sine/cosine pairs.

    Captures:
      - Diurnal (24-h) solar/thermal forcing cycle.
      - Annual (365-day) seasonality driving swell climatology.
    """
    hour       = df.index.hour
    doy        = df.index.dayofyear
    df = df.copy()
    df['hour_sin']       = np.sin(2 * np.pi * hour / 24.0)
    df['hour_cos']       = np.cos(2 * np.pi * hour / 24.0)
    df['doy_sin']        = np.sin(2 * np.pi * doy  / 365.0)
    df['doy_cos']        = np.cos(2 * np.pi * doy  / 365.0)
    return df


def add_physics_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Domain-informed derived features.

    Wave Steepness
    --------------
    S = H_s / L_0  where L_0 = g·T_p² / (2π) is the deep-water wavelength.
    High steepness (> ~1/7) signals wave breaking; physically bounds model
    output and represents a nonlinear interaction between Hs and Tp.

    Wind Momentum Components
    ------------------------
    U_x = V_wind · sin(θ)   (eastward projection)
    U_y = V_wind · cos(θ)   (northward projection)
    Captures the vector nature of wind forcing on wave generation,
    preserving directional information lost by scalar wind speed alone.
    """
    df = df.copy()

    # ── Target buoy physics ───────────────────────────────────────────────
    if ('target_buoy_hs' in df.columns and 'target_buoy_tp' in df.columns):
        denom = G * df['target_buoy_tp']**2 / (2 * np.pi)
        df['target_buoy_wave_steepness'] = (
            df['target_buoy_hs'] / denom.replace(0, np.nan)
        )

    if ('target_buoy_windspeed' in df.columns and
            'target_buoy_winddir' in df.columns):
        theta = np.deg2rad(df['target_buoy_winddir'])
        df['target_buoy_wind_mom_x'] = df['target_buoy_windspeed'] * np.sin(theta)
        df['target_buoy_wind_mom_y'] = df['target_buoy_windspeed'] * np.cos(theta)

    # ── Offshore point physics ─────────────────────────────────────────────
    for pid in OFFSHORE_IDS:
        hs_col    = f"{pid}_hs"
        tp_col    = f"{pid}_tp"
        wsp_col   = f"{pid}_windspeed"
        wdir_col  = f"{pid}_winddirection"

        if hs_col in df.columns and tp_col in df.columns:
            denom = G * df[tp_col]**2 / (2 * np.pi)
            df[f"{pid}_wave_steepness"] = (
                df[hs_col] / denom.replace(0, np.nan)
            )

        if wsp_col in df.columns and wdir_col in df.columns:
            theta = np.deg2rad(df[wdir_col])
            df[f"{pid}_wind_mom_x"] = df[wsp_col] * np.sin(theta)
            df[f"{pid}_wind_mom_y"] = df[wsp_col] * np.cos(theta)

    return df


print("  ✓ Lag features function        : add_lag_features()")
print("  ✓ Rolling stats function       : add_rolling_features()")
print("  ✓ Cyclical encoding function   : add_cyclical_time_features()")
print("  ✓ Physics features function    : add_physics_features()")


STEP F: FEATURE ENGINEERING FUNCTIONS
  ✓ Lag features function        : add_lag_features()
  ✓ Rolling stats function       : add_rolling_features()
  ✓ Cyclical encoding function   : add_cyclical_time_features()
  ✓ Physics features function    : add_physics_features()


### 8 — Independent Feature Engineering

In [9]:
print("\n" + "="*60)
print("STEP G: APPLYING FEATURES INDEPENDENTLY TO EACH SET")
print("="*60)

# Columns to receive lag + rolling treatment
# (all continuous predictors: target buoy history + all offshore vars)
def get_feature_cols(df: pd.DataFrame) -> list:
    """Return all numeric columns that exist in df (exclude target)."""
    exclude = {'target'}
    return [c for c in df.select_dtypes(include=[np.number]).columns
            if c not in exclude]


def engineer(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Full feature engineering pipeline for a single base dataframe."""
    print(f"\n  [{label}] Base shape: {df.shape}")

    # ── FIX: Drop columns that are entirely NaN BEFORE engineering ──────────
    # Root cause: buoy SSH is set to np.nan in Step 1 (no real SSH data at
    # coastal buoys). Lag/rolling features derived from an all-NaN column are
    # also 100% NaN, and a subsequent dropna() would wipe every row.
    # We audit and remove such columns here so they never enter the pipeline.
    all_nan_cols = [c for c in df.columns if df[c].isna().all()]
    if all_nan_cols:
        df = df.drop(columns=all_nan_cols)
        print(f"  [{label}] Dropped {len(all_nan_cols)} all-NaN cols "
              f"(e.g. {all_nan_cols[:5]})")

    # ── FIX: Also drop columns with >95 % NaN (sparse offshore vars) ────────
    high_nan_cols = [c for c in df.columns
                     if df[c].isna().mean() > 0.95 and c != 'target']
    if high_nan_cols:
        df = df.drop(columns=high_nan_cols)
        print(f"  [{label}] Dropped {len(high_nan_cols)} >95%-NaN cols "
              f"(e.g. {high_nan_cols[:5]})")

    feat_cols = get_feature_cols(df)

    df = add_lag_features(df, feat_cols, LAG_STEPS)
    print(f"  [{label}] After lags:    {df.shape}")

    df = add_rolling_features(df, feat_cols, ROLLING_STEP_SIZES)
    print(f"  [{label}] After rolling: {df.shape}")

    df = add_cyclical_time_features(df)
    print(f"  [{label}] After cyclical: {df.shape}")

    df = add_physics_features(df)
    print(f"  [{label}] After physics: {df.shape}")

    return df


df_train_eng = engineer(df_train_base, "TRAIN")
df_oos_eng   = engineer(df_oos_base,   "OOS")

del df_train_base, df_oos_base, df_offshore_wide
gc.collect()


STEP G: APPLYING FEATURES INDEPENDENTLY TO EACH SET

  [TRAIN] Base shape: (12482, 36)
  [TRAIN] Dropped 1 all-NaN cols (e.g. ['target_buoy_ssh'])
  [TRAIN] After lags:    (12482, 385)
  [TRAIN] After rolling: (12482, 700)
  [TRAIN] After cyclical: (12482, 704)
  [TRAIN] After physics: (12482, 722)

  [OOS] Base shape: (5609, 36)
  [OOS] Dropped 1 all-NaN cols (e.g. ['target_buoy_ssh'])
  [OOS] After lags:    (5609, 385)
  [OOS] After rolling: (5609, 700)
  [OOS] After cyclical: (5609, 704)
  [OOS] After physics: (5609, 722)


0

### 9 — Target Shift & Final dropna()

In [10]:
print("\n" + "="*60)
print("STEP H: TARGET SHIFT (-1 STEP = -3H AHEAD) & DROPNA")
print("="*60)

# Create target column: Hs at t+3h (predict one step into the future)
df_train_eng['target'] = df_train_eng['target_buoy_hs'].shift(-FORECAST_LEAD_STEPS)
df_oos_eng['target']   = df_oos_eng['target_buoy_hs'].shift(-FORECAST_LEAD_STEPS)

# ── Targeted dropna strategy ─────────────────────────────────────────────────
# We drop ONLY on the subset of columns that must be non-NaN for a valid sample:
#   • 'target'            — the label we are trying to predict
#   • 'target_buoy_hs'    — the primary observation (used as lag_0)
#   • 'target_buoy_tp'    — second buoy observable
# Engineered lag/rolling features can have NaN at series boundaries (first
# 10 steps of each set, or after temporal gaps) but those NaN cells are
# legitimate structural zeros for the model — we do NOT purge those rows.
# Rows where the raw observation itself is missing are genuinely unusable.
BASE_DROP_COLS = ['target', 'target_buoy_hs']
# Add tp if it exists in both sets
if 'target_buoy_tp' in df_train_eng.columns:
    BASE_DROP_COLS.append('target_buoy_tp')

# ── TRAIN ────────────────────────────────────────────────────────────────────
n_before_train = len(df_train_eng)
# Step 1: drop rows where essential base observations are NaN
df_train_eng.dropna(subset=BASE_DROP_COLS, inplace=True)
# Step 2: fill remaining NaN in engineered features (lag/rolling boundary NaNs)
#         using forward-fill then backward-fill — safe because values are
#         temporally local and the fill distance is at most 10 steps (30 h).
df_train_eng.ffill(inplace=True)
df_train_eng.bfill(inplace=True)
# Step 3: hard drop any residual NaN (e.g. features at absolute start of series)
df_train_eng.dropna(inplace=True)
n_after_train = len(df_train_eng)
print(f"  TRAIN  rows before dropna: {n_before_train:,}")
print(f"  TRAIN  rows after  dropna: {n_after_train:,}  "
      f"({n_before_train - n_after_train:,} removed)")

# ── OOS ──────────────────────────────────────────────────────────────────────
n_before_oos = len(df_oos_eng)
df_oos_eng.dropna(subset=BASE_DROP_COLS, inplace=True)
df_oos_eng.ffill(inplace=True)
df_oos_eng.bfill(inplace=True)
df_oos_eng.dropna(inplace=True)
n_after_oos = len(df_oos_eng)
print(f"  OOS    rows before dropna: {n_before_oos:,}")
print(f"  OOS    rows after  dropna: {n_after_oos:,}  "
      f"({n_before_oos - n_after_oos:,} removed)")

# Sanity checks
assert n_after_train > 0, "FATAL: 0 rows in Train after dropna – check merge keys!"
assert n_after_oos   > 0, "FATAL: 0 rows in OOS after dropna – check merge keys!"
print("\n  ✓ Both sets are non-empty. Row counts verified.")


STEP H: TARGET SHIFT (-1 STEP = -3H AHEAD) & DROPNA
  TRAIN  rows before dropna: 12,482
  TRAIN  rows after  dropna: 12,477  (5 removed)
  OOS    rows before dropna: 5,609
  OOS    rows after  dropna: 5,605  (4 removed)

  ✓ Both sets are non-empty. Row counts verified.


### 10 — Separate X / y

In [11]:
print("\n" + "="*60)
print("STEP I: SEPARATING X / y")
print("="*60)

TARGET_COL   = 'target'

X_train = df_train_eng.drop(columns=[TARGET_COL])
y_train = df_train_eng[[TARGET_COL]]

X_oos   = df_oos_eng.drop(columns=[TARGET_COL])
y_oos   = df_oos_eng[[TARGET_COL]]

# Align OOS to train feature columns (drop any OOS-only extras; fill missing with NaN)
X_oos = X_oos.reindex(columns=X_train.columns, fill_value=np.nan)

print(f"  X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"  X_oos   : {X_oos.shape}     y_oos   : {y_oos.shape}")
print(f"  Feature count: {X_train.shape[1]}")


STEP I: SEPARATING X / y
  X_train : (12477, 722)   y_train : (12477, 1)
  X_oos   : (5605, 722)     y_oos   : (5605, 1)
  Feature count: 722


### 11 — Spearman Correlation Heatmap (Top 15 Features vs Target)

In [12]:
print("\n" + "="*60)
print("STEP J: SPEARMAN CORRELATION HEATMAP")
print("="*60)

set_pub_style()

# Compute Spearman correlations of all X features against target in Train
train_for_corr = X_train.copy()
train_for_corr[TARGET_COL] = y_train[TARGET_COL]

spearman_corrs = (
    train_for_corr
    .corr(method='spearman')[TARGET_COL]
    .drop(labels=[TARGET_COL])
    .dropna()
)

# Select top 15 by absolute correlation magnitude
top15_idx = spearman_corrs.abs().nlargest(15).index
top15_corr = spearman_corrs[top15_idx]

# Build a (15×1) correlation matrix for heatmap aesthetics
heatmap_data = top15_corr.to_frame(name='ρ (Spearman vs. target Hs+3h)')

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    linewidths=0.4,
    linecolor='white',
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={'shrink': 0.7, 'label': 'Spearman ρ'}
)
ax.set_title(
    'Top 15 Feature Correlations with Target\n'
    r'(Spearman $\rho$, Train Set)',
    fontsize=13,
    pad=10
)
ax.set_xlabel('')
ax.set_ylabel('Feature', fontsize=11)
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9)
sns.despine(fig=fig, left=True, bottom=True)
plt.tight_layout()

save_fig(fig, 'spearman_top15_features_vs_target')
print("  ✓ Spearman heatmap saved.")


STEP J: SPEARMAN CORRELATION HEATMAP


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/spearman_top15_features_vs_target.png


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/spearman_top15_features_vs_target.tiff
  ✓ Spearman heatmap saved.


### 12 — Timeline Density Plot with 24-h Gap Annotation

In [13]:
print("\n" + "="*60)
print("STEP K: TIMELINE DENSITY PLOT")
print("="*60)

set_pub_style()

fig, ax = plt.subplots(figsize=(14, 4))

# Resample to daily observation counts
daily_train = (
    df_train_eng.resample('1D').count().iloc[:, 0]
    .clip(upper=1)    # presence/absence per day
)
daily_oos = (
    df_oos_eng.resample('1D').count().iloc[:, 0]
    .clip(upper=1)
)

ax.fill_between(daily_train.index, daily_train.values,
                alpha=0.55, color='steelblue', label='Train / Validation')
ax.fill_between(daily_oos.index, daily_oos.values,
                alpha=0.55, color='darkorange', label='Out-of-Sample (OOS)')

# Annotate the 24-hour gap
gap_start = df_train_eng.index.max()
gap_end   = df_oos_eng.index.min()
ax.axvspan(gap_start, gap_end, color='crimson', alpha=0.20,
           label='24-h Anti-Leakage Gap')
ax.axvline(gap_start, color='crimson', linewidth=1.2, linestyle='--')
ax.axvline(gap_end,   color='crimson', linewidth=1.2, linestyle='--')
ax.annotate(
    f'24-h gap\n({gap_start.strftime("%Y-%m-%d")} →\n{gap_end.strftime("%Y-%m-%d")})',
    xy=((gap_start + (gap_end - gap_start) / 2), 0.55),
    xycoords=('data', 'axes fraction'),
    ha='center', fontsize=9, color='crimson',
    arrowprops=None
)

ax.set_xlim(daily_train.index.min(), daily_oos.index.max())
ax.set_ylim(0, 1.25)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Data Presence (1=observed)', fontsize=11)
ax.set_title(
    'Train / OOS Temporal Coverage with Anti-Leakage Gap\n'
    '(Purges 12.4-h tidal M2 autocorrelation & 24-h thermal breeze inertia)',
    fontsize=12
)
ax.legend(fontsize=10, framealpha=0.9)
sns.despine(fig=fig)
plt.tight_layout()

save_fig(fig, 'timeline_train_oos_coverage')
print("  ✓ Timeline plot saved.")


STEP K: TIMELINE DENSITY PLOT


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/timeline_train_oos_coverage.png


    ✓ Saved → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/timeline_train_oos_coverage.tiff
  ✓ Timeline plot saved.


### 13 — Export CSVs

In [14]:
print("\n" + "="*60)
print("STEP L: EXPORTING CSV FILES")
print("="*60)

exports = {
    'X_train_KBS.csv' : X_train,
    'y_train_KBS.csv' : y_train,
    'X_oos_KBS.csv'   : X_oos,
    'y_oos_KBS.csv'   : y_oos,
}

for fname, df_export in exports.items():
    out_path = os.path.join(OUTPUT_DIR, fname)
    df_export.to_csv(out_path, date_format='%Y-%m-%d %H:%M:%S')
    print(f"  ✓ {fname:<22}  shape: {df_export.shape}  → {out_path}")


STEP L: EXPORTING CSV FILES
  ✓ X_train_KBS.csv         shape: (12477, 722)  → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/X_train_KBS.csv
  ✓ y_train_KBS.csv         shape: (12477, 1)  → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/y_train_KBS.csv
  ✓ X_oos_KBS.csv           shape: (5605, 722)  → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/X_oos_KBS.csv
  ✓ y_oos_KBS.csv           shape: (5605, 1)  → /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/y_oos_KBS.csv


### 14 — Pipeline Summary

In [15]:
print("\n" + "="*70)
print("  STEP 2 FEATURE ENGINEERING PIPELINE — COMPLETE")
print("="*70)
print(f"  X_train : {X_train.shape[0]:>7,} rows × {X_train.shape[1]:>4} features")
print(f"  y_train : {y_train.shape[0]:>7,} rows")
print(f"  X_oos   : {X_oos.shape[0]:>7,} rows × {X_oos.shape[1]:>4} features")
print(f"  y_oos   : {y_oos.shape[0]:>7,} rows")
print(f"  Output  : {OUTPUT_DIR}")
print("="*70)


  STEP 2 FEATURE ENGINEERING PIPELINE — COMPLETE
  X_train :  12,477 rows ×  722 features
  y_train :  12,477 rows
  X_oos   :   5,605 rows ×  722 features
  y_oos   :   5,605 rows
  Output  : /content/drive/MyDrive/KBS_Paper/Outputs/2_Feature_Engineering_KBS/
